# Tooth-AI paper evaluation

Runs Tasks D and E on the 716-image stratified test split.

**Run the cells in order.** Cell 1 must show a GPU or everything below is
meaningless: the Kaggle push API has been reported to hand out a different
machine than the one requested without raising, so this notebook checks rather
than trusts.


## 1. GPU gate — fails loudly if there is no GPU


In [ ]:
import os, subprocess, sys

# Pin to ONE card before anything imports torch. The Kaggle box has 2x T4 and
# nothing in this harness is multi-GPU, so an unpinned run makes the latency
# figure ambiguous about which card produced it.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

try:
    rows = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,uuid,memory.total",
         "--format=csv,noheader"], text=True).strip().splitlines()
except Exception as exc:
    raise SystemExit(f"FATAL: no nvidia-smi -- this session has no GPU ({exc})")
if not rows:
    raise SystemExit("FATAL: nvidia-smi returned no GPU")

print("visible cards:")
for r in rows:
    print("   ", r)
print()
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
PINNED = rows[0].strip()
print("latency will be attributed to:", PINNED)
print("\nThis exact string goes into every results CSV as the latency hardware.")


## 2. Install the pinned stack

The checkpoints were trained and used with **torch 2.5.1+cu121** and
**Detectron2 0.6 at commit `fd27788`** (read from the project's own
`dist-info`/`direct_url.json`). We try those first.

Any deviation prints a loud warning — small numeric differences follow from a
different torch build, and the paper must not silently report numbers from an
unpinned stack.

This cell is deliberately alone so it can be re-run and debugged by itself.


In [ ]:
import subprocess, sys, warnings

PIN_TORCH        = "2.5.1"
PIN_TV           = "0.20.1"
PIN_CUDA         = "cu121"
PIN_D2_COMMIT    = "fd27788985af0f4ca800bca563acdb700bb890e2"

def sh(cmd):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, text=True).returncode

DEVIATIONS = []

# --- torch -----------------------------------------------------------------
rc = sh([sys.executable, "-m", "pip", "install", "-q",
         f"torch=={PIN_TORCH}+{PIN_CUDA}", f"torchvision=={PIN_TV}+{PIN_CUDA}",
         "--index-url", f"https://download.pytorch.org/whl/{PIN_CUDA}"])
if rc != 0:
    DEVIATIONS.append(f"torch {PIN_TORCH}+{PIN_CUDA} would not install; kept the preinstalled build")
    print("!! pinned torch failed to install -- continuing with whatever is present")

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("FATAL: torch cannot see the GPU")
if not torch.__version__.startswith(PIN_TORCH):
    DEVIATIONS.append(f"torch is {torch.__version__}, pinned {PIN_TORCH}+{PIN_CUDA}")


### 2b. Detectron2 at the pinned commit


In [ ]:
# Build the exact commit the checkpoints were produced with. If that build
# fails (CUDA/toolchain drift on the Kaggle image), fall back to main and say so.
rc = sh([sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/facebookresearch/detectron2.git@{PIN_D2_COMMIT}"])
if rc != 0:
    DEVIATIONS.append(f"Detectron2 {PIN_D2_COMMIT[:7]} would not build; used main instead")
    print("!! pinned Detectron2 commit failed to build -- falling back to main")
    rc = sh([sys.executable, "-m", "pip", "install", "-q",
             "git+https://github.com/facebookresearch/detectron2.git"])
    if rc != 0:
        raise SystemExit("FATAL: Detectron2 would not build at all")

sh([sys.executable, "-m", "pip", "install", "-q",
    "opencv-python-headless", "pycocotools"])


### 2c. Verify — and shout about any deviation


In [ ]:
import torch, detectron2, cv2, numpy, pycocotools
print("torch      ", torch.__version__)
print("detectron2 ", detectron2.__version__)
print("opencv     ", cv2.__version__)
print("numpy      ", numpy.__version__)
print("GPU        ", torch.cuda.get_device_name(0))
print("device_count", torch.cuda.device_count(), "(must be 1 -- pinned)")
assert torch.cuda.device_count() == 1, (
    "CUDA_VISIBLE_DEVICES did not take; latency would be ambiguous across "
    f"{torch.cuda.device_count()} cards")

PIN_NUMPY = "2.2.6"      # from the project's own dist-info
if not numpy.__version__.startswith(PIN_NUMPY):
    DEVIATIONS.append(f"numpy is {numpy.__version__}, pinned {PIN_NUMPY} "
                      "(Kaggle's preinstalled build; not overridden because "
                      "forcing it risks breaking the image's torch)")

from detectron2.engine import DefaultPredictor   # import must actually work
print("\ndetectron2 imports cleanly")

if DEVIATIONS:
    print("\n" + "!" * 72)
    print("VERSION DEVIATION FROM THE PINNED STACK")
    for d in DEVIATIONS:
        print("  -", d)
    print("Numbers from this run may differ slightly from the pinned stack.")
    print("Record this in the paper's reproducibility note.")
    print("!" * 72)
else:
    print("\nStack matches the pins exactly.")




## 3. Locate the dataset

Attach the private Kaggle Dataset built by `make_bundle.py`. It appears under
`/kaggle/input/<slug>/`.


In [ ]:
import os, glob, shutil, sys

# Two datasets are attached on purpose:
#   tooth-ai-paper-eval       ~2 GB   data/test + output/<model>/  (never changes)
#   tooth-ai-paper-eval-code  ~200 KB the harness                  (changes often)
# Splitting them makes a harness fix a 2-second upload, not 14 minutes.
#
# Do NOT assume a layout. Two things vary:
#   * mount point:  /kaggle/input/<slug>/  vs  /kaggle/input/datasets/<user>/<name>/
#   * zip handling: a bundle with several top-level folders keeps their names,
#     but a bundle with a single folder gets FLATTENED, so the code dataset's
#     files land at the dataset root with no paper_eval/ prefix.
# So locate things by a file that must exist, not by a directory name.
def find_file(fname, depth_max=7):
    hits = []
    for d in range(1, depth_max):
        hits += [h for h in glob.glob("/kaggle/input/" + "*/" * d + fname)
                 if os.path.isfile(h)]
    return sorted(set(hits))

def find_dir(sub, depth_max=7):
    hits = []
    for d in range(1, depth_max):
        hits += [h for h in glob.glob("/kaggle/input/" + "*/" * d + sub)
                 if os.path.isdir(h)]
    return sorted(set(hits))

code_hits = find_file("run_eval.py")          # the harness lives beside this
data_hits = find_dir("data/test")
out_hits  = find_dir("output")

if not (code_hits and data_hits and out_hits):
    print("Could not locate every input. Tree under /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.count("/") > 7:
            dirs[:] = []
            continue
        print("  ", root, "dirs:", sorted(dirs)[:8], "files:", sorted(files)[:6])
    raise SystemExit(f"FATAL: run_eval.py={bool(code_hits)} data/test={bool(data_hits)} "
                     f"output={bool(out_hits)}. Attach BOTH private datasets.")

# Prefer the code-only dataset: the one whose run_eval.py has no data/ sibling.
def pick_code(hits):
    for h in hits:
        root = os.path.dirname(h)
        anc = os.path.dirname(root)
        if not os.path.isdir(os.path.join(root, "data")) and \
           not os.path.isdir(os.path.join(anc, "data")):
            return root
    return os.path.dirname(hits[0])

CODE = pick_code(code_hits)
DATA = data_hits[0]
OUT  = out_hits[0]
print("harness from :", CODE)
print("data/test    :", DATA)
print("output       :", OUT)

WORK = "/kaggle/working/tooth"
os.makedirs(WORK, exist_ok=True)
dst = os.path.join(WORK, "paper_eval")          # results are written here
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(CODE, dst)
assert os.path.exists(os.path.join(dst, "run_eval.py")), "harness copy failed"

os.makedirs(os.path.join(WORK, "data/final-di-stratified"), exist_ok=True)
os.makedirs(os.path.join(WORK, "data/final-di-remapped"), exist_ok=True)
# 9-class annotations for the same 716 images, if the bundle carries them
# 9-class annotations: shipped in the CODE dataset (7.8 MB) to avoid
# re-uploading the 2 GB data bundle. Fall back to the data bundle if present.
nine_src = None
for cand in (os.path.join(os.path.dirname(CODE), "annotations9"),
             os.path.join(CODE, "..", "annotations9"),
             os.path.join(os.path.dirname(DATA), "test9")):
    if os.path.isdir(cand):
        nine_src = os.path.abspath(cand); break
nine_dst = os.path.join(WORK, "data/final-di-remapped/test")
if nine_src and not os.path.exists(nine_dst):
    os.symlink(nine_src, nine_dst)
print("9-class annotations:", nine_src or "ABSENT -- that model's COCO eval will be skipped")
for link, target in ((os.path.join(WORK, "data/final-di-stratified/test"), DATA),
                     (os.path.join(WORK, "output"), OUT)):
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.exists(link):
        shutil.rmtree(link)
    os.symlink(target, link)

os.chdir(WORK)
print("\ncwd:", os.getcwd())

# exFAT AppleDouble sidecars rode into the 2 GB dataset and "._x.jpg" matches *.jpg.
jpgs = [f for f in glob.glob("data/final-di-stratified/test/*.jpg")
        if not os.path.basename(f).startswith("._")]
print("test images:", len(jpgs), "(expected 716)")
assert len(jpgs) == 716, f"expected 716 test images, found {len(jpgs)}"

import json as _json
meta = _json.load(open("data/final-di-stratified/test/_annotations.coco.json"))
print("annotations:", len(meta["images"]), "images,", len(meta["annotations"]), "anns")

for m in ("resnet50_9class_20k", "rtx4060_48k", "resnext101_cascade_60k"):
    w = f"output/{m}/model_final.pth"
    print(f"  {m:<26} " + (f"OK {os.path.getsize(w)/1e6:.0f} MB" if os.path.exists(w)
                            else "MISSING"))

# Catch a missing module-level import here, not 20 minutes into a GPU run.
print()
assert os.system("python3 paper_eval/check_imports.py") == 0, "check_imports failed"




## 4a. Routing A/B — does the app.py:953-958 bug touch the teeth list?

The bug routes an anomaly detection from the 9-class model into the **teeth**
list. It cannot fire on an image where that model finds no anomaly, and the 93
full-arch images carry zero anomaly annotations — so the ordinary smoke sample
cannot settle it.

This samples from the 380 images that do carry anomaly labels and diffs the
teeth lists both ways. **If they are identical, the audit has to say the bug
does not affect tooth metrics.**


In [ ]:
# Routing A/B already settled: teeth lists differ on 12/12 anomaly-bearing
# images. Skipping to keep this run short.
RERUN_ROUTING_AB = False
if RERUN_ROUTING_AB:
    !python3 paper_eval/run_eval.py --routing-ab
else:
    print("SKIPPED (routing A/B) -- settled at 12/12 on the previous run.")


## 4. Smoke test — 5 images

Proves the whole harness runs before spending an hour on the full split.
Check the printed table is populated, then move on.


In [ ]:
# FULL RUN. The 5-image smoke test passed on kernel v4, so the heavy cells
# below are live. Stages write their CSVs as they complete and the ablation
# runs first, so a session that dies mid-way still leaves finished work on disk.
RUN_FULL = True

# Skipped on a full run -- the ablation cell below covers the same configs at
# full scale. Set RUN_FULL=False to use this as a 5-image smoke test again.
if RUN_FULL:
    print("SKIPPED (5-image smoke test) -- RUN_FULL is True; see the full ablation below.")
else:
    !python3 paper_eval/run_eval.py --all --smoke-test --latency


## 5. Full run — all configurations


In [ ]:
# Ablation already complete (v6) and per_image.csv is in hand -- skipping.
RERUN_ABLATION = False
if RERUN_ABLATION:
    !python3 paper_eval/run_eval.py --all
else:
    print("SKIPPED (ablation) -- results/ablation.csv and per_image.csv from v6 stand.")


## 6. Threshold sweep


In [ ]:
# Sweep already completed and nothing affecting it changed -- skipping.
# Set RERUN_SWEEP = True to redo it (~16 min).
RERUN_SWEEP = False
if RERUN_SWEEP:
    !python3 paper_eval/run_eval.py --sweep
else:
    print("SKIPPED (tau sweep) -- results/threshold_sweep.csv from the prior run stands.")


## 7. COCO re-evaluation on the 716 split

Three points: R50 9-class, R50 41-class, X101 Cascade 41-class — each at its
own configured test size, which the CSV records.


In [ ]:
RERUN_COCO = False
if RERUN_COCO:
    !python3 paper_eval/run_eval.py --coco-eval
else:
    print("SKIPPED (COCO eval) -- results/coco_eval_716.csv from v6 stands.")

# THIS RUN: sweep tau for the network baseline so the head-to-head compares
# geometry against the network's OWN optimum, not against a threshold chosen
# for a different model. Inference is cached per image; only tau varies.
!python3 paper_eval/run_eval.py --numbering-sweep


## 8. Figure candidates and panels


In [ ]:
RERUN_CANDIDATES = False
if RERUN_CANDIDATES:
    !python3 paper_eval/export_panels.py --suggest-images
else:
    print("SKIPPED (panel candidates) -- prior run found 0/67 with any central-zone gain.")


In [ ]:
# Pick an image_id from the table above -- ideally one flagged as introducing
# a central-zone false positive -- and export its panels.
IMAGE_ID = None          # <-- set me

if IMAGE_ID is None:
    print("set IMAGE_ID from the candidate table above, then re-run this cell")
else:
    !python3 paper_eval/export_panels.py --image-id {IMAGE_ID}


## 9. Zip the results


In [ ]:
import shutil, os
out = "/kaggle/working/paper_eval_results"
shutil.make_archive(out, "zip", "paper_eval/results")
print(out + ".zip", os.path.getsize(out + ".zip"), "bytes")
for root, _d, files in os.walk("paper_eval/results"):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.getsize(p):>10}  {p}")
